In [1]:
# Memeriksa GPU yang tersedia
!nvidia-smi

Tue Nov 24 03:14:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install "numpy>=1.22" "scapy>=2.5.0" "scikit-learn>=1.1" "torch>=2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 55.1 MB/s  0:00:00


In [3]:
import numpy as np
import scapy
import sklearn
import torch

print("NumPy Version       :", np.__version__)
print("Scapy Version       :", scapy.__version__)
print("Scikit-learn Version:", sklearn.__version__)
print("PyTorch Version     :", torch.__version__)

print("\nCUDA Available      :", torch.cuda.is_available())

NumPy Version       : 2.4.6
Scapy Version       : 2.8.0
Scikit-learn Version: 1.6.1
PyTorch Version     : 2.14.1+cu130

CUDA Available      : True


## Modul Pemuatan Data (data_loader.py)

In [4]:
%%writefile data_loader.py
"""
Data loading and preprocessing for SepSpace experiments.
Supports: USTC-TFC2016, MCFP, ISCX-VPN, ISCX-nonVPN.
"""

import os
import json
import glob
import random
import numpy as np
from scapy.all import rdpcap, IP
from collections import defaultdict

CACHE_ROOT = os.path.join("data", "preprocessed")


def pcap_to_units(pcap_path, max_units, window_l=8, stride_s=4, mtu=1500):
    """
    Extract Channel Units from a single PCAP file.

    Args:
        pcap_path: Path to PCAP file
        max_units: Maximum number of Channel Units to extract
        window_l: Channel Unit length (number of packets)
        stride_s: Sliding window stride
        mtu: Maximum transmission unit for length clipping

    Returns:
        List of Channel Units, each as a list of token IDs
    """
    try:
        pkts = rdpcap(pcap_path)
    except Exception as e:
        print(f"  [WARN] Cannot read {pcap_path}: {e}")
        return []

    # Build (direction, length) sequence
    seq = []
    first_src = None
    for pkt in pkts:
        if not pkt.haslayer(IP):
            continue
        src = pkt[IP].src
        if first_src is None:
            first_src = src
        direction = 0 if pkt[IP].src == first_src else 1
        length = min(len(pkt), mtu)
        seq.append((direction, length))
        if len(seq) >= max_units * window_l * 2:
            break

    if len(seq) < window_l:
        return []

    # Sliding window → token IDs
    N_RESERVED = 2  # 0=PAD, 1=CLS
    units = []
    i = 0
    while i + window_l <= len(seq) and len(units) < max_units:
        window = seq[i : i + window_l]
        tokens = [N_RESERVED + length + direction * mtu for direction, length in window]
        units.append(tokens)
        i += stride_s

    return units


def load_ustc_tfc2016(root, n_cap, window_l=8, stride_s=4, mtu=1500, seed=42):
    """
    Load USTC-TFC2016 dataset with preprocessing cache.

    Returns:
        X: np.array of shape (N, window_l), dtype=int32
        y: np.array of shape (N,), dtype=int64
        class_names: List[str]
        vocab_size: int
    """
    # Check cache
    cache_dir = os.path.join(CACHE_ROOT, "ustc")
    cache_path = os.path.join(cache_dir, "features.npy")
    meta_path = os.path.join(cache_dir, "meta.json")

    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            meta = json.load(f)

        # Verify preprocessing parameters match
        if (
            meta.get("n_cap") == n_cap
            and meta.get("window_l") == window_l
            and meta.get("stride_s") == stride_s
            and meta.get("mtu") == mtu
        ):

            print(f"Loading from cache: {cache_dir}")
            X = np.load(os.path.join(cache_dir, "features.npy"))
            y = np.load(os.path.join(cache_dir, "labels.npy"))

            with open(os.path.join(cache_dir, "class_stats.json"), "r") as f:
                stats = json.load(f)

            return X, y, stats["class_names"], stats["vocab_size"]

    # No cache, do full preprocessing
    malware_dir = os.path.join(root, "Malware")
    benign_dir = os.path.join(root, "Benign")

    # Exclude adware adload and wannaCry
    exclude_classes = ["adware adload", "wannaCry"]

    # Merge split files
    merge_map = {
        "SMB-1": "SMB",
        "SMB-2": "SMB",
        "Weibo-1": "Weibo",
        "Weibo-2": "Weibo",
        "Weibo-3": "Weibo",
        "Weibo-4": "Weibo",
    }

    class_files = []
    for pcap in sorted(glob.glob(os.path.join(malware_dir, "*.pcap"))):
        name = os.path.basename(pcap).replace(".pcap", "")
        if name in exclude_classes:
            print(f"  [EXCLUDE] M-{name}: extra class not in original USTC")
            continue
        class_files.append((pcap, f"M-{name}"))
    for pcap in sorted(glob.glob(os.path.join(benign_dir, "*.pcap"))):
        name = os.path.basename(pcap).replace(".pcap", "")
        if name in exclude_classes:
            print(f"  [EXCLUDE] B-{name}: extra class not in original USTC")
            continue
        class_files.append((pcap, f"B-{name}"))

    print(f"Loading USTC-TFC2016 from {root}")

    # Aggregate units by merged class name
    class_units = defaultdict(list)
    for pcap_path, class_name in class_files:
        base_name = class_name.split("-", 1)[1]  # Remove M-/B- prefix
        merged_name = merge_map.get(base_name, base_name)
        merged_class = f"{class_name[0]}-{merged_name}"  # Restore M-/B- prefix

        units = pcap_to_units(
            pcap_path,
            max_units=n_cap * 2,
            window_l=window_l,
            stride_s=stride_s,
            mtu=mtu,
        )
        class_units[merged_class].extend(units)
        print(f"  {class_name} → {merged_class}: {len(units)} units")

    # Sample from merged classes
    all_units, all_labels, class_names = [], [], []
    for label_idx, (class_name, units) in enumerate(sorted(class_units.items())):
        n = min(len(units), n_cap)
        if n < 200:
            print(f"  [SKIP] {class_name}: only {n} units (< 200 threshold)")
            continue

        if len(units) > n_cap:
            random.seed(seed)
            units = random.sample(units, n_cap)
        else:
            units = units[:n]

        print(f"  {class_name}: {n} units (final)")
        all_units.extend(units)
        all_labels.extend([label_idx] * n)
        class_names.append(class_name)

    # Pad to window_l
    vocab_size = 2 + 2 * mtu + 1
    padded = []
    for u in all_units:
        if len(u) < window_l:
            u = u + [0] * (window_l - len(u))
        padded.append(u[:window_l])

    X = np.array(padded, dtype=np.int32)
    y = np.array(all_labels, dtype=np.int64)

    # Save to cache
    os.makedirs(cache_dir, exist_ok=True)
    np.save(os.path.join(cache_dir, "features.npy"), X)
    np.save(os.path.join(cache_dir, "labels.npy"), y)

    with open(os.path.join(cache_dir, "class_stats.json"), "w") as f:
        json.dump(
            {
                "class_names": class_names,
                "vocab_size": vocab_size,
                "class_counts": {
                    name: int(np.sum(y == i)) for i, name in enumerate(class_names)
                },
            },
            f,
            indent=2,
        )

    with open(meta_path, "w") as f:
        json.dump(
            {
                "n_cap": n_cap,
                "window_l": window_l,
                "stride_s": stride_s,
                "mtu": mtu,
                "seed": seed,
                "root": root,
                "timestamp": str(np.datetime64("now")),
            },
            f,
            indent=2,
        )

    print(f"Saved to cache: {cache_dir}")

    return X, y, class_names, vocab_size


def load_mcfp(root, n_cap, window_l=8, stride_s=4, mtu=1500, seed=42):
    """
    Load MCFP dataset (8 malware families) with preprocessing cache.
    """
    cache_dir = os.path.join(CACHE_ROOT, "mcfp")
    cache_path = os.path.join(cache_dir, "features.npy")
    meta_path = os.path.join(cache_dir, "meta.json")

    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            meta = json.load(f)
        if (
            meta.get("n_cap") == n_cap
            and meta.get("window_l") == window_l
            and meta.get("stride_s") == stride_s
            and meta.get("mtu") == mtu
        ):
            print(f"Loading from cache: {cache_dir}")
            X = np.load(os.path.join(cache_dir, "features.npy"))
            y = np.load(os.path.join(cache_dir, "labels.npy"))
            with open(os.path.join(cache_dir, "class_stats.json"), "r") as f:
                stats = json.load(f)
            return X, y, stats["class_names"], stats["vocab_size"]

    families = [
        "Cobalt",
        "Dridex",
        "Normal-32",
        "TrickBot",
        "Trojan_Downloader",
        "Trojan_Locky",
        "Worm_Allape",
        "Zeus",
    ]

    print(f"Loading MCFP from {root}")
    all_units, all_labels, class_names = [], [], []

    for label_idx, family in enumerate(families):
        family_dir = os.path.join(root, family)
        if not os.path.isdir(family_dir):
            print(f"  [SKIP] {family}: directory not found")
            continue

        pcaps = glob.glob(os.path.join(family_dir, "*.pcap")) + glob.glob(
            os.path.join(family_dir, "*.pcapng")
        )

        family_units = []
        for pcap in pcaps:
            units = pcap_to_units(
                pcap, max_units=n_cap * 2, window_l=window_l, stride_s=stride_s, mtu=mtu
            )
            family_units.extend(units)
            if len(family_units) >= n_cap * 2:
                break

        n = min(len(family_units), n_cap)
        if n < 200:
            print(f"  [SKIP] {family}: only {n} units (< 200 threshold)")
            continue

        if len(family_units) > n_cap:
            random.seed(seed)
            family_units = random.sample(family_units, n_cap)
        else:
            family_units = family_units[:n]

        print(f"  {family}: {n} units")
        all_units.extend(family_units)
        all_labels.extend([label_idx] * n)
        class_names.append(family)

    vocab_size = 2 + 2 * mtu + 1
    padded = []
    for u in all_units:
        if len(u) < window_l:
            u = u + [0] * (window_l - len(u))
        padded.append(u[:window_l])

    X = np.array(padded, dtype=np.int32)
    y = np.array(all_labels, dtype=np.int64)

    os.makedirs(cache_dir, exist_ok=True)
    np.save(os.path.join(cache_dir, "features.npy"), X)
    np.save(os.path.join(cache_dir, "labels.npy"), y)

    with open(os.path.join(cache_dir, "class_stats.json"), "w") as f:
        json.dump(
            {
                "class_names": class_names,
                "vocab_size": vocab_size,
                "class_counts": {
                    name: int(np.sum(y == i)) for i, name in enumerate(class_names)
                },
            },
            f,
            indent=2,
        )

    with open(meta_path, "w") as f:
        json.dump(
            {
                "n_cap": n_cap,
                "window_l": window_l,
                "stride_s": stride_s,
                "mtu": mtu,
                "seed": seed,
                "root": root,
                "timestamp": str(np.datetime64("now")),
            },
            f,
            indent=2,
        )

    print(f"Saved to cache: {cache_dir}")
    return X, y, class_names, vocab_size


def _vpn_app(basename):
    b = basename.replace("vpn_", "").replace(".pcap", "").replace(".pcapng", "")
    if b.startswith("aim"):
        return "aim"
    if b.startswith("facebook"):
        return "facebook"
    if b.startswith("hangouts"):
        return "hangouts"
    if b.startswith("icq"):
        return "icq"
    if b.startswith("netflix"):
        return "netflix"
    if b.startswith("skype"):
        return "skype"
    if b.startswith("youtube"):
        return "youtube"
    if b.startswith("spotify"):
        return "spotify"
    if b.startswith("vimeo"):
        return "vimeo"
    if b.startswith("ftps"):
        return "ftps"
    if b.startswith("sftp"):
        return "sftp"
    if b.startswith("email"):
        return "email"
    if b.startswith("voipbuster"):
        return "voipbuster"
    if b.startswith("bittorrent"):
        return "bittorrent"
    return None


def _vpn_service(basename):
    b = basename.replace("vpn_", "").replace(".pcap", "").replace(".pcapng", "")
    if b.startswith("email"):
        return "email"
    if b.startswith("aim_chat"):
        return "chat"
    if b.startswith("facebook_chat"):
        return "chat"
    if b.startswith("hangouts_chat"):
        return "chat"
    if b.startswith("icq_chat"):
        return "chat"
    if (
        b.startswith("netflix")
        or b.startswith("youtube")
        or b.startswith("vimeo")
        or b.startswith("spotify")
    ):
        return "streaming"
    if b.startswith("ftps") or b.startswith("sftp"):
        return "file_transfer"
    if (
        b.startswith("facebook_audio")
        or b.startswith("hangouts_audio")
        or b.startswith("skype_audio")
        or b.startswith("voipbuster")
    ):
        return "voip"
    if b.startswith("bittorrent"):
        return "p2p"
    return None


def _nonvpn_app(basename):
    b = basename.replace(".pcap", "").replace(".pcapng", "")
    if b.startswith("aimchat") or b.startswith("aim_chat"):
        return "aim"
    if b.startswith("email"):
        return "email"
    if b.startswith("facebook"):
        return "facebook"
    if b.startswith("hangouts") or b.startswith("hangout_"):
        return "hangouts"
    if b.startswith("icqchat") or b.startswith("icq_chat"):
        return "icq"
    if b.startswith("netflix"):
        return "netflix"
    if b.startswith("skype"):
        return "skype"
    if b.startswith("youtube"):
        return "youtube"
    if b.startswith("spotify"):
        return "spotify"
    if b.startswith("vimeo"):
        return "vimeo"
    if b.startswith("voipbuster"):
        return "voipbuster"
    if b.startswith("ftps"):
        return "ftps"
    if (
        b.startswith("sftp")
        or b.startswith("sftpdown")
        or b.startswith("sftpup")
        or b.startswith("sftp_")
    ):
        return "sftp"
    if b.startswith("scp") or b.startswith("scpdown") or b.startswith("scpup"):
        return "scp"
    return None


def _nonvpn_service(basename):
    b = basename.replace(".pcap", "").replace(".pcapng", "")
    if b.startswith("email"):
        return "email"
    if b.startswith("aimchat") or b.startswith("aim_chat"):
        return "chat"
    if b.startswith("icqchat") or b.startswith("icq_chat"):
        return "chat"
    if b.startswith("facebookchat") or b.startswith("facebook_chat"):
        return "chat"
    if b.startswith("hangout_chat") or b.startswith("hangouts_chat"):
        return "chat"
    if b.startswith("netflix"):
        return "streaming"
    if b.startswith("youtube") and not b.startswith("youtubehtml"):
        return "streaming"
    if b.startswith("ftps"):
        return "file_transfer"
    if (
        b.startswith("sftp")
        or b.startswith("sftpdown")
        or b.startswith("sftpup")
        or b.startswith("sftp_")
    ):
        return "file_transfer"
    if b.startswith("scp") or b.startswith("scpdown") or b.startswith("scpup"):
        return "file_transfer"
    if b.startswith("facebook_audio"):
        return "voip"
    if b.startswith("hangouts_audio"):
        return "voip"
    return None


_ISCX_LABEL_FN = {
    ("VPN", "app"): _vpn_app,
    ("VPN", "service"): _vpn_service,
    ("nonVPN", "app"): _nonvpn_app,
    ("nonVPN", "service"): _nonvpn_service,
}


def load_iscx(
    root,
    subset="nonVPN",
    label_scheme="app",
    n_cap=5000,
    window_l=8,
    stride_s=4,
    mtu=1500,
    seed=42,
):
    assert subset in ("VPN", "nonVPN")
    assert label_scheme in ("app", "service")

    cache_key = f"iscx_{subset.lower()}_{label_scheme}"
    cache_dir = os.path.join(CACHE_ROOT, cache_key)
    meta_path = os.path.join(cache_dir, "meta.json")

    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            meta = json.load(f)
        if (
            meta.get("n_cap") == n_cap
            and meta.get("window_l") == window_l
            and meta.get("stride_s") == stride_s
            and meta.get("mtu") == mtu
            and meta.get("label_scheme") == label_scheme
        ):
            print(f"Loading from cache: {cache_dir}")
            X = np.load(os.path.join(cache_dir, "features.npy"))
            y = np.load(os.path.join(cache_dir, "labels.npy"))
            with open(os.path.join(cache_dir, "class_stats.json"), "r") as f:
                stats = json.load(f)
            return X, y, stats["class_names"], stats["vocab_size"]

    if subset == "VPN":
        dirs = [os.path.join(root, "VPN-PCAPS-01"), os.path.join(root, "VPN-PCAPs-02")]
    else:
        dirs = [
            os.path.join(root, d)
            for d in ["NonVPN-PCAPs-01", "NonVPN-PCAPs-02", "NonVPN-PCAPs-03"]
        ]

    label_fn = _ISCX_LABEL_FN[(subset, label_scheme)]
    print(f"Loading ISCX-{subset} ({label_scheme}-level) from {root}")

    class_pcaps = defaultdict(list)
    skipped = []
    for d in dirs:
        if not os.path.isdir(d):
            continue
        for pcap in sorted(
            glob.glob(os.path.join(d, "*.pcap"))
            + glob.glob(os.path.join(d, "*.pcapng"))
        ):
            label = label_fn(os.path.basename(pcap).lower())
            if label is None:
                skipped.append(os.path.basename(pcap))
            else:
                class_pcaps[label].append(pcap)

    if skipped:
        print(f"  [SKIP] {len(skipped)} uncertain files")

    class_units = defaultdict(list)
    for cls, pcaps in sorted(class_pcaps.items()):
        for pcap in pcaps:
            u = pcap_to_units(
                pcap, max_units=n_cap * 2, window_l=window_l, stride_s=stride_s, mtu=mtu
            )
            class_units[cls].extend(u)
            if len(class_units[cls]) >= n_cap * 2:
                break
        print(f"  {cls}: {len(class_units[cls])} units from {len(pcaps)} pcap(s)")

    counts = {cls: len(units) for cls, units in class_units.items()}
    valid_classes = sorted(cls for cls, cnt in counts.items() if cnt >= 500)
    print(f"  Valid classes (≥500): {len(valid_classes)} — {valid_classes}")

    if len(valid_classes) < 5:
        print(f"  [WARN] Only {len(valid_classes)} valid classes")

    actual_n_cap = min(n_cap, min((counts[c] for c in valid_classes), default=0))
    print(f"  Balanced N_cap: {actual_n_cap}")

    all_units, all_labels, class_names = [], [], []
    for label_idx, cls in enumerate(valid_classes):
        units = class_units[cls]
        if len(units) > actual_n_cap:
            random.seed(seed)
            units = random.sample(units, actual_n_cap)
        else:
            units = units[:actual_n_cap]
        all_units.extend(units)
        all_labels.extend([label_idx] * len(units))
        class_names.append(cls)

    vocab_size = 2 + 2 * mtu + 1
    padded = []
    for u in all_units:
        if len(u) < window_l:
            u = u + [0] * (window_l - len(u))
        padded.append(u[:window_l])

    X = np.array(padded, dtype=np.int32)
    y = np.array(all_labels, dtype=np.int64)

    os.makedirs(cache_dir, exist_ok=True)
    np.save(os.path.join(cache_dir, "features.npy"), X)
    np.save(os.path.join(cache_dir, "labels.npy"), y)

    with open(os.path.join(cache_dir, "class_stats.json"), "w") as f:
        json.dump(
            {
                "class_names": class_names,
                "vocab_size": vocab_size,
                "class_counts": {
                    name: int(np.sum(y == i)) for i, name in enumerate(class_names)
                },
                "n_valid_classes": len(valid_classes),
                "actual_n_cap": actual_n_cap,
                "skipped_files": skipped,
            },
            f,
            indent=2,
        )

    with open(meta_path, "w") as f:
        json.dump(
            {
                "n_cap": n_cap,
                "window_l": window_l,
                "stride_s": stride_s,
                "mtu": mtu,
                "seed": seed,
                "root": root,
                "subset": subset,
                "label_scheme": label_scheme,
                "timestamp": str(np.datetime64("now")),
            },
            f,
            indent=2,
        )

    print(f"Saved to cache: {cache_dir}")
    return X, y, class_names, vocab_size


def split_data(X, y, novel_idx, val_ratio=0.2, test_ratio=0.2, seed=42):
    np.random.seed(seed)

    known_mask = y != novel_idx
    novel_mask = y == novel_idx

    Xk, yk = X[known_mask], y[known_mask]
    Xn, yn = X[novel_mask], y[novel_mask]

    known_classes = sorted(set(yk.tolist()))
    remap = {c: i for i, c in enumerate(known_classes)}
    yk_r = np.array([remap[c] for c in yk])

    n = len(yk_r)
    idx = np.random.permutation(n)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)

    test_idx = idx[:n_test]
    val_idx = idx[n_test : n_test + n_val]
    train_idx = idx[n_test + n_val :]

    X_train, y_train = Xk[train_idx], yk_r[train_idx]
    X_val, y_val = Xk[val_idx], yk_r[val_idx]
    X_test_k, y_test_k = Xk[test_idx], yk_r[test_idx]

    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test_k,
        y_test_k,
        Xn,
        yn,
        len(known_classes),
    )


Writing data_loader.py


In [5]:
with open('/content/data_loader.py', 'r') as f:
    lines = f.readlines()
print(''.join(lines[-25:]))


    n = len(yk_r)
    idx = np.random.permutation(n)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)

    test_idx = idx[:n_test]
    val_idx = idx[n_test : n_test + n_val]
    train_idx = idx[n_test + n_val :]

    X_train, y_train = Xk[train_idx], yk_r[train_idx]
    X_val, y_val = Xk[val_idx], yk_r[val_idx]
    X_test_k, y_test_k = Xk[test_idx], yk_r[test_idx]

    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test_k,
        y_test_k,
        Xn,
        yn,
        len(known_classes),
    )



## Modul Pendukung / Baseline (nnfst.py)

In [6]:
%%writefile nnfst.py
"""
nNFST: Nearest-class Null-space Feature Subspace Transform (no novel prior).

Adaptation of NFST for open-set recognition without novel-class prior.
Only known-class training embeddings are used to build per-class subspaces.
Novelty score = minimum reconstruction error across all known-class subspaces.

Reference: Zhang et al., "Towards Open Set Deep Networks" .
"""

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

def compute_extra_metrics(nov_k, nov_n, correct, total):
    """Fungsi cadangan internal untuk menghitung metrik tambahan tanpa membutuhkan modul eksternal."""
    # Hitung metrik tambahan dasar secara lokal
    return {"closed_set_correct": correct, "closed_set_total": total}

class NNFSTClassifier:
    """
    No-prior NFST classifier.

    For each known class k, compute the within-class subspace via PCA on
    centered class embeddings. Novelty score for a test point x is:

        score(x) = min_k  ||x - proj_k(x)||^2

    where proj_k(x) is the projection of x onto class k's subspace.
    Lower score = more likely known; higher score = more likely novel.

    Threshold tau_k is calibrated on the validation set at the 95th percentile
    of known-class reconstruction errors (zero novel samples used).
    """

    def __init__(self, n_components=None, var_ratio=0.95):
        """
        Args:
            n_components: fixed number of PCA components per class.
                          If None, use var_ratio to select automatically.
            var_ratio: cumulative variance ratio to retain (used when n_components=None).
        """
        self.n_components = n_components
        self.var_ratio = var_ratio
        self.subspaces = []   # List of (mean_k, V_k) per class
        self.n_classes = 0

    def fit(self, h_train: np.ndarray, y_train: np.ndarray):
        """
        Build per-class PCA subspaces from training embeddings.

        Args:
            h_train: (N, d) L2-normalized embeddings
            y_train: (N,) integer class labels 0..K-1
        """
        self.n_classes = int(y_train.max()) + 1
        self.subspaces = []

        for k in range(self.n_classes):
            mask = y_train == k
            H_k = h_train[mask]  # (N_k, d)

            if len(H_k) < 2:
                # Degenerate: store identity subspace
                self.subspaces.append((np.zeros(h_train.shape[1]), None))
                continue

            mean_k = H_k.mean(axis=0)
            H_centered = H_k - mean_k  # (N_k, d)

            # SVD for PCA
            U, S, Vt = np.linalg.svd(H_centered, full_matrices=False)
            # Vt: (min(N_k,d), d) — rows are principal components

            if self.n_components is not None:
                n_comp = min(self.n_components, Vt.shape[0])
            else:
                # Select components by cumulative variance
                var = S ** 2
                cum_var = np.cumsum(var) / (var.sum() + 1e-12)
                n_comp = int(np.searchsorted(cum_var, self.var_ratio)) + 1
                n_comp = min(n_comp, Vt.shape[0])

            V_k = Vt[:n_comp]  # (n_comp, d) — top principal components

            self.subspaces.append((mean_k, V_k))

    def reconstruction_error(self, h: np.ndarray) -> np.ndarray:
        """
        Compute per-class reconstruction errors.

        Args:
            h: (N, d) embeddings

        Returns:
            errors: (N, K) reconstruction errors per class
        """
        N, d = h.shape
        errors = np.zeros((N, self.n_classes))

        for k, (mean_k, V_k) in enumerate(self.subspaces):
            if V_k is None:
                # Degenerate class: use full distance from mean
                errors[:, k] = np.sum((h - mean_k) ** 2, axis=1)
                continue

            h_centered = h - mean_k  # (N, d)
            # Project onto subspace: proj = h_centered @ V_k.T @ V_k
            coords = h_centered @ V_k.T  # (N, n_comp)
            proj = coords @ V_k          # (N, d)
            residual = h_centered - proj  # (N, d)
            errors[:, k] = np.sum(residual ** 2, axis=1)

        return errors

    def novelty_score(self, h: np.ndarray) -> np.ndarray:
        """
        Novelty score = min reconstruction error across all known classes.

        Args:
            h: (N, d) embeddings

        Returns:
            scores: (N,) novelty scores (higher = more novel)
        """
        errors = self.reconstruction_error(h)
        return errors.min(axis=1)

    def predict_class(self, h: np.ndarray) -> np.ndarray:
        """
        Closed-set prediction: nearest class by reconstruction error.

        Args:
            h: (N, d) embeddings

        Returns:
            pred: (N,) predicted class indices
        """
        errors = self.reconstruction_error(h)
        return errors.argmin(axis=1)

    def calibrate_threshold(self, h_val: np.ndarray, y_val: np.ndarray,
                            fpr_target: float = 0.05) -> float:
        """
        Calibrate global threshold at (1 - fpr_target) quantile of known-class
        novelty scores on validation set. Zero novel samples used.

        Args:
            h_val: (N, d) validation embeddings
            y_val: (N,) validation labels (all known classes)
            fpr_target: target FPR (default 0.05 → 95th percentile)

        Returns:
            tau: float threshold
        """
        scores = self.novelty_score(h_val)
        tau = float(np.quantile(scores, 1.0 - fpr_target))
        return tau


def evaluate_nnfst(model, train_loader, val_loader, test_k_loader, novel_loader,
                   device, n_components=None, var_ratio=0.95):
    """
    Full nNFST evaluation pipeline.

    Steps:
    1. Extract L2-normalized embeddings from encoder (no SupCon training needed)
    2. Fit per-class PCA subspaces on training embeddings
    3. Calibrate threshold on validation set
    4. Evaluate on test set (known + novel)

    Args:
        model: trained BaselineModel or SepSpaceModel (CE-only Stage 1)
        train_loader, val_loader, test_k_loader, novel_loader: DataLoaders
        device: torch device
        n_components: PCA components per class (None = auto by var_ratio)
        var_ratio: cumulative variance to retain

    Returns:
        results: dict {auroc, dr, acc, n1}
    """
    from models import get_embeddings

    model.eval()

    # Extract embeddings
    h_train, y_train = get_embeddings(model, train_loader, device)
    h_val, y_val = get_embeddings(model, val_loader, device)
    h_test_k, y_test_k = get_embeddings(model, test_k_loader, device)
    h_novel, _ = get_embeddings(model, novel_loader, device)

    h_train_np = h_train.numpy()
    y_train_np = y_train.numpy()
    h_val_np = h_val.numpy()
    y_val_np = y_val.numpy()
    h_test_k_np = h_test_k.numpy()
    h_novel_np = h_novel.numpy()

    # Fit nNFST
    clf = NNFSTClassifier(n_components=n_components, var_ratio=var_ratio)
    clf.fit(h_train_np, y_train_np)

    # Calibrate threshold
    tau = clf.calibrate_threshold(h_val_np, y_val_np, fpr_target=0.05)

    # Novelty scores on test set
    nov_k = clf.novelty_score(h_test_k_np)
    nov_n = clf.novelty_score(h_novel_np)

    nov_all = np.concatenate([nov_k, nov_n])
    gt = np.concatenate([np.zeros(len(nov_k)), np.ones(len(nov_n))])

    # AUROC
    auroc = roc_auc_score(gt, nov_all)

    # DR: fraction of novel samples with score > tau
    dr = float((nov_n > tau).mean())

    # Known-class accuracy (closed-set prediction on test_k)
    pred_k = clf.predict_class(h_test_k_np)
    correct = int((pred_k == y_test_k.numpy()).sum())
    total = len(y_test_k)
    acc = correct / total

    n1 = 2 * acc * dr / (acc + dr + 1e-8)

    extra = compute_extra_metrics(nov_k, nov_n, correct, total)

    return {"auroc": auroc, "dr": dr, "acc": acc, "n1": n1, **extra}


Writing nnfst.py


## Modul Evaluasi (evaluate.py)

In [7]:
%%writefile evaluate.py
"""
Evaluation methods for all baselines and proposed method.
Includes: MSP, Energy, Mahalanobis, prototype-radius, LOF.
"""

import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EmpiricalCovariance


def compute_extra_metrics(nov_scores_known, nov_scores_novel, closed_set_correct, closed_set_total):
    """
    Compute FPR95, AUPR, and OSCR from novelty scores.

    Args:
        nov_scores_known: np.ndarray, novelty scores for known-class test samples (higher = more novel)
        nov_scores_novel: np.ndarray, novelty scores for novel-class test samples
        closed_set_correct: int, number of correctly classified known-class samples
        closed_set_total: int, total known-class test samples

    Returns:
        dict with keys: fpr95, aupr, oscr
    """
    # FPR95: FPR on known-class samples when TPR on novel = 95%
    gt = np.concatenate([np.zeros(len(nov_scores_known)), np.ones(len(nov_scores_novel))])
    scores = np.concatenate([nov_scores_known, nov_scores_novel])

    # Sort thresholds descending (higher score = more novel)
    sorted_thresholds = np.sort(scores)[::-1]
    best_fpr95 = 1.0
    for tau in sorted_thresholds:
        tpr = (nov_scores_novel >= tau).mean()
        fpr = (nov_scores_known >= tau).mean()
        if tpr >= 0.95:
            best_fpr95 = fpr
            break

    # AUPR: area under precision-recall curve (novel class = positive)
    aupr = average_precision_score(gt, scores)

    # OSCR: Open-Set Classification Rate
    # = fraction of known samples correctly classified AND not rejected
    # at threshold where novel TPR = 95%
    tau95 = np.quantile(nov_scores_novel, 0.05)  # 95% of novel above this
    known_not_rejected = (nov_scores_known < tau95).sum()
    oscr = (closed_set_correct / closed_set_total) * (known_not_rejected / len(nov_scores_known)) if closed_set_total > 0 else 0.0

    return {"fpr95": float(best_fpr95), "aupr": float(aupr), "oscr": float(oscr)}


@torch.no_grad()
def evaluate_prototype_radius(model, test_k_loader, novel_loader, protos, taus, device):
    """
    Evaluate CE+SupCon+PR (prototype-radius rejection).

    Returns:
        results: dict with keys {auroc, dr, acc, n1}
    """
    from models import get_embeddings, predict_open_set

    model.eval()
    h_test_k, y_test_k = get_embeddings(model, test_k_loader, device)
    h_novel, _ = get_embeddings(model, novel_loader, device)

    # Combine known + novel
    h_all = torch.cat([h_test_k, h_novel])
    gt_novel = torch.cat([
        torch.zeros(len(h_test_k), dtype=torch.long),
        torch.ones(len(h_novel), dtype=torch.long)
    ])

    pred, novelty_score = predict_open_set(h_all, protos, taus)

    # AUROC
    auroc = roc_auc_score(gt_novel.numpy(), novelty_score.numpy())

    # DR: fraction of novel samples correctly flagged as unknown
    novel_pred = pred[len(h_test_k):]
    dr = (novel_pred == -1).float().mean().item()

    # Known-class accuracy (excluding samples predicted as unknown)
    known_pred = pred[:len(h_test_k)]
    known_correct = (known_pred == y_test_k).float()
    known_classified = known_pred != -1
    if known_classified.sum() > 0:
        acc = known_correct[known_classified].mean().item()
    else:
        acc = 0.0

    n1 = 2 * acc * dr / (acc + dr + 1e-8)

    nov_k_np = novelty_score[:len(h_test_k)].numpy()
    nov_n_np = novelty_score[len(h_test_k):].numpy()
    correct_count = int(known_correct[known_classified].sum().item()) if known_classified.sum() > 0 else 0
    extra = compute_extra_metrics(nov_k_np, nov_n_np, correct_count, len(h_test_k))

    return {"auroc": auroc, "dr": dr, "acc": acc, "n1": n1, **extra}


@torch.no_grad()
def evaluate_msp(model, test_k_loader, novel_loader, device):
    """
    Evaluate CE+MSP (Maximum Softmax Probability baseline).

    Returns:
        results: dict with keys {auroc, dr, acc, n1}
    """
    model.eval()

    # Collect MSP scores
    def get_msp(loader):
        scores, labels = [], []
        for X, y in loader:
            X = X.to(device)
            logits, _ = model.classify(X)
            prob = F.softmax(logits, dim=-1)
            msp = prob.max(dim=-1).values
            scores.append(msp.cpu())
            labels.append(y)
        return torch.cat(scores), torch.cat(labels)

    msp_k, yk = get_msp(test_k_loader)
    msp_n, _ = get_msp(novel_loader)

    # Novelty score = 1 - MSP (higher = more novel)
    nov_k = 1.0 - msp_k
    nov_n = 1.0 - msp_n
    nov_all = torch.cat([nov_k, nov_n])
    gt = torch.cat([torch.zeros(len(nov_k)), torch.ones(len(nov_n))])

    # AUROC
    auroc = roc_auc_score(gt.numpy(), nov_all.numpy())

    # Threshold: 95th percentile of known novelty scores
    tau_msp = torch.quantile(nov_k, 0.95).item()
    dr = (nov_n > tau_msp).float().mean().item()

    # Known-class accuracy
    correct, total = 0, 0
    for X, y in test_k_loader:
        X, y = X.to(device), y.to(device)
        logits, _ = model.classify(X)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    acc = correct / total if total > 0 else 0.0

    n1 = 2 * acc * dr / (acc + dr + 1e-8)

    extra = compute_extra_metrics(nov_k.numpy(), nov_n.numpy(), correct, total)

    return {"auroc": auroc, "dr": dr, "acc": acc, "n1": n1, **extra}


@torch.no_grad()
def evaluate_energy(model, test_k_loader, novel_loader, device):
    """
    Evaluate CE+Energy (Liu et al. NeurIPS 2020).
    Energy score = log(sum(exp(logits)))

    Returns:
        results: dict with keys {auroc, dr, acc, n1}
    """
    model.eval()

    def get_energy(loader):
        scores, labels = [], []
        for X, y in loader:
            X = X.to(device)
            logits, _ = model.classify(X)
            energy = torch.logsumexp(logits, dim=-1)
            scores.append(energy.cpu())
            labels.append(y)
        return torch.cat(scores), torch.cat(labels)

    energy_k, yk = get_energy(test_k_loader)
    energy_n, _ = get_energy(novel_loader)

    # Novelty score = -energy (higher = more novel)
    nov_k = -energy_k
    nov_n = -energy_n
    nov_all = torch.cat([nov_k, nov_n])
    gt = torch.cat([torch.zeros(len(nov_k)), torch.ones(len(nov_n))])

    # AUROC
    auroc = roc_auc_score(gt.numpy(), nov_all.numpy())

    # Threshold: 95th percentile
    tau_energy = torch.quantile(nov_k, 0.95).item()
    dr = (nov_n > tau_energy).float().mean().item()

    # Known-class accuracy
    correct, total = 0, 0
    for X, y in test_k_loader:
        X, y = X.to(device), y.to(device)
        logits, _ = model.classify(X)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    acc = correct / total if total > 0 else 0.0

    n1 = 2 * acc * dr / (acc + dr + 1e-8)

    extra = compute_extra_metrics(nov_k.numpy(), nov_n.numpy(), correct, total)

    return {"auroc": auroc, "dr": dr, "acc": acc, "n1": n1, **extra}


@torch.no_grad()
def evaluate_mahalanobis(model, train_loader, val_loader, test_k_loader, novel_loader, device):
    """
    Evaluate CE+Mahalanobis (Lee et al. NeurIPS 2018).
    Class-conditional Gaussian distance in feature space.

    Returns:
        results: dict with keys {auroc, dr, acc, n1}
    """
    from models import get_embeddings

    model.eval()

    # Get training embeddings (not L2-normalized for Mahalanobis)
    h_train_list, y_train_list = [], []
    for X, y in train_loader:
        X = X.to(device)
        _, h = model.classify(X)
        h_train_list.append(h.cpu())
        y_train_list.append(y)
    h_train = torch.cat(h_train_list)
    y_train = torch.cat(y_train_list)

    n_classes = y_train.max().item() + 1

    # Compute class means and shared covariance
    class_means = []
    for k in range(n_classes):
        mask = y_train == k
        class_means.append(h_train[mask].mean(0))
    class_means = torch.stack(class_means)  # (K, d)

    # Shared covariance (pooled)
    centered = []
    for k in range(n_classes):
        mask = y_train == k
        centered.append(h_train[mask] - class_means[k])
    centered = torch.cat(centered)
    cov = EmpiricalCovariance().fit(centered.numpy())
    precision = torch.tensor(cov.precision_, dtype=torch.float32)

    def mahalanobis_dist(h, means, prec):
        """Compute Mahalanobis distance to each class."""
        # h: (N, d), means: (K, d), prec: (d, d)
        # dist[i, k] = (h[i] - means[k])^T @ prec @ (h[i] - means[k])
        dists = []
        for k in range(means.shape[0]):
            diff = h - means[k]  # (N, d)
            dist_k = (diff @ prec * diff).sum(dim=-1)  # (N,)
            dists.append(dist_k)
        return torch.stack(dists, dim=1)  # (N, K)

    # Get test embeddings
    h_test_k, y_test_k = [], []
    for X, y in test_k_loader:
        X = X.to(device)
        _, h = model.classify(X)
        h_test_k.append(h.cpu())
        y_test_k.append(y)
    h_test_k = torch.cat(h_test_k)
    y_test_k = torch.cat(y_test_k)

    h_novel = []
    for X, y in novel_loader:
        X = X.to(device)
        _, h = model.classify(X)
        h_novel.append(h.cpu())
    h_novel = torch.cat(h_novel)

    # Compute distances
    dists_k = mahalanobis_dist(h_test_k, class_means, precision)
    dists_n = mahalanobis_dist(h_novel, class_means, precision)

    # Novelty score = min Mahalanobis distance (higher = more novel)
    nov_k = dists_k.min(dim=1).values
    nov_n = dists_n.min(dim=1).values
    nov_all = torch.cat([nov_k, nov_n])
    gt = torch.cat([torch.zeros(len(nov_k)), torch.ones(len(nov_n))])

    # AUROC
    auroc = roc_auc_score(gt.numpy(), nov_all.numpy())

    # Threshold: 95th percentile
    tau_maha = torch.quantile(nov_k, 0.95).item()
    dr = (nov_n > tau_maha).float().mean().item()

    # Known-class accuracy
    correct, total = 0, 0
    for X, y in test_k_loader:
        X, y = X.to(device), y.to(device)
        logits, _ = model.classify(X)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    acc = correct / total if total > 0 else 0.0

    n1 = 2 * acc * dr / (acc + dr + 1e-8)

    extra = compute_extra_metrics(nov_k.numpy(), nov_n.numpy(), correct, total)

    return {"auroc": auroc, "dr": dr, "acc": acc, "n1": n1, **extra}


@torch.no_grad()
def evaluate_lof(model, train_loader, test_k_loader, novel_loader, device, n_neighbors=20):
    """
    Evaluate CE+SupCon+LOF (Local Outlier Factor).

    Returns:
        results: dict with keys {auroc, dr, acc, n1}
    """
    from models import get_embeddings

    model.eval()

    # Get embeddings
    h_train, y_train = get_embeddings(model, train_loader, device)
    h_test_k, y_test_k = get_embeddings(model, test_k_loader, device)
    h_novel, _ = get_embeddings(model, novel_loader, device)

    # Fit LOF on training set
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=True)
    lof.fit(h_train.numpy())

    # Predict novelty scores (negative_outlier_factor)
    nov_k = -lof.score_samples(h_test_k.numpy())
    nov_n = -lof.score_samples(h_novel.numpy())
    nov_all = np.concatenate([nov_k, nov_n])
    gt = np.concatenate([np.zeros(len(nov_k)), np.ones(len(nov_n))])

    # AUROC
    auroc = roc_auc_score(gt, nov_all)

    # Threshold: 95th percentile
    tau_lof = np.quantile(nov_k, 0.95)
    dr = (nov_n > tau_lof).mean()

    # Known-class accuracy
    correct, total = 0, 0
    for X, y in test_k_loader:
        X, y = X.to(device), y.to(device)
        logits, _ = model.classify(X)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    acc = correct / total if total > 0 else 0.0

    n1 = 2 * acc * dr / (acc + dr + 1e-8)

    extra = compute_extra_metrics(nov_k, nov_n, correct, total)

    return {"auroc": auroc, "dr": float(dr), "acc": acc, "n1": n1, **extra}


def print_results(results, tag):
    """Pretty-print evaluation results."""
    print(f"\n[{tag}]")
    print(f"  AUROC : {results['auroc']:.4f}")
    print(f"  DR    : {results['dr']:.4f}")
    print(f"  Acc   : {results['acc']:.4f}")
    print(f"  N1    : {results['n1']:.4f}")
    if "fpr95" in results:
        print(f"  FPR95 : {results['fpr95']:.4f}")
        print(f"  AUPR  : {results['aupr']:.4f}")
        print(f"  OSCR  : {results['oscr']:.4f}")


Writing evaluate.py


## Arsitektur Model (models.py)

In [8]:
%%writefile models.py
"""
Model definitions for SepSpace experiments.
Includes: Channel Encoder, SepSpace model, baseline heads.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ChannelEncoder(nn.Module):
    """Transformer-based Channel Encoder from prior work."""

    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, seq_len=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = nn.Embedding(seq_len + 1, d_model)  # +1 for CLS
        enc_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=d_model * 2,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.d_model = d_model

    def forward(self, x):
        """
        Args:
            x: (B, L) token IDs
        Returns:
            h: (B, d_model) CLS token embedding
        """
        B, L = x.shape
        cls_tok = torch.ones(B, 1, dtype=torch.long, device=x.device)
        x = torch.cat([cls_tok, x], dim=1)  # (B, L+1)
        pos_ids = torch.arange(L + 1, device=x.device).unsqueeze(0)
        h = self.embed(x) + self.pos(pos_ids)
        h = self.transformer(h)
        return h[:, 0]  # CLS token


class SepSpaceModel(nn.Module):
    """
    SepSpace: CE+SupCon with prototype-radius rejection.

    Components:
    - encoder: Channel Encoder
    - classifier: Linear classifier (for CE loss)
    - proj_head: 2-layer MLP projection head (for SupCon loss, training only)
    """

    def __init__(self, vocab_size, n_classes, d_model=64, d_proj=32):
        super().__init__()
        self.encoder = ChannelEncoder(vocab_size, d_model)
        self.classifier = nn.Linear(d_model, n_classes)
        self.proj_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_proj)
        )

    def forward(self, x):
        """Returns encoder embedding (for inference)."""
        return self.encoder(x)

    def classify(self, x):
        """Returns (logits, embedding)."""
        h = self.encoder(x)
        return self.classifier(h), h

    def project(self, x):
        """Returns L2-normalized projection (for SupCon)."""
        h = self.encoder(x)
        z = self.proj_head(h)
        return F.normalize(z, dim=-1)


class BaselineModel(nn.Module):
    """
    Baseline model: CE-only training.
    Used for MSP, Energy, Mahalanobis baselines.
    """

    def __init__(self, vocab_size, n_classes, d_model=64):
        super().__init__()
        self.encoder = ChannelEncoder(vocab_size, d_model)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        """Returns encoder embedding."""
        return self.encoder(x)

    def classify(self, x):
        """Returns (logits, embedding)."""
        h = self.encoder(x)
        return self.classifier(h), h


def supcon_loss(z, labels, temp=0.07):
    """
    Supervised Contrastive Loss (Khosla et al. NeurIPS 2020).

    Args:
        z: (B, d) L2-normalized embeddings
        labels: (B,) class labels
        temp: temperature

    Returns:
        loss: scalar
    """
    B = z.shape[0]
    sim = torch.mm(z, z.T) / temp  # (B, B)

    # Mask out self
    mask_self = torch.eye(B, dtype=torch.bool, device=z.device)

    # Positive mask: same label, not self
    labels = labels.view(-1, 1)
    mask_pos = (labels == labels.T) & ~mask_self  # (B, B)

    # Numerical stability
    sim_max, _ = sim.max(dim=1, keepdim=True)
    sim = sim - sim_max.detach()

    exp_sim = torch.exp(sim)
    exp_sim_no_self = exp_sim * (~mask_self).float()
    log_prob = sim - torch.log(exp_sim_no_self.sum(dim=1, keepdim=True) + 1e-8)

    # Mean over positives
    n_pos = mask_pos.float().sum(dim=1)
    loss = -(log_prob * mask_pos.float()).sum(dim=1)
    valid = n_pos > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=z.device)
    loss = (loss[valid] / n_pos[valid]).mean()
    return loss


def train_stage1(model, loader, epochs, lr, device):
    """
    Stage 1: CE-only training.

    Args:
        model: SepSpaceModel or BaselineModel
        loader: DataLoader
        epochs: number of epochs
        lr: learning rate
        device: torch device
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    for ep in range(epochs):
        total_loss, correct, total = 0, 0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits, _ = model.classify(X)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)

        if (ep + 1) % 5 == 0:
            acc = correct / total
            print(f"  Stage1 ep{ep+1:02d}: loss={total_loss/len(loader):.4f} acc={acc:.4f}")


def train_stage2(model, loader, epochs, lr, lambda_sup, device):
    """
    Stage 2: CE + SupCon joint fine-tuning.
    Classifier frozen, encoder + proj_head trainable.

    Args:
        model: SepSpaceModel
        loader: DataLoader
        epochs: number of epochs
        lr: learning rate (typically 0.3 * stage1_lr)
        lambda_sup: weight for SupCon loss
        device: torch device
    """
    # Freeze classifier
    for p in model.classifier.parameters():
        p.requires_grad = False

    opt = torch.optim.Adam(
        list(model.encoder.parameters()) + list(model.proj_head.parameters()),
        lr=lr
    )
    model.train()

    for ep in range(epochs):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits, _ = model.classify(X)
            loss_ce = F.cross_entropy(logits, y)
            z = model.project(X)
            loss_sc = supcon_loss(z, y)
            loss = loss_ce + lambda_sup * loss_sc

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        if (ep + 1) % 5 == 0:
            print(f"  Stage2 ep{ep+1:02d}: loss={total_loss/len(loader):.4f}")

    # Unfreeze classifier
    for p in model.classifier.parameters():
        p.requires_grad = True


@torch.no_grad()
def get_embeddings(model, loader, device):
    """
    Extract L2-normalized embeddings from encoder.

    Returns:
        h: (N, d) embeddings
        y: (N,) labels
    """
    model.eval()
    hs, ys = [], []
    for X, y in loader:
        X = X.to(device)
        _, h = model.classify(X)
        hs.append(F.normalize(h, dim=-1).cpu())
        ys.append(y)
    return torch.cat(hs), torch.cat(ys)


def compute_prototypes(h_train, y_train, n_classes):
    """
    Compute class prototypes as L2-normalized mean of training embeddings.

    Returns:
        protos: (K, d) prototypes
    """
    protos = []
    for k in range(n_classes):
        mask = y_train == k
        if mask.sum() == 0:
            # Empty class (shouldn't happen in practice)
            protos.append(torch.zeros(h_train.shape[1]))
        else:
            proto = h_train[mask].mean(0)
            protos.append(F.normalize(proto, dim=0))
    return torch.stack(protos)  # (K, d)


def cosine_dist(h, protos):
    """
    Compute cosine distance: d(h, μ) = 1 - h·μ

    Args:
        h: (N, d) embeddings
        protos: (K, d) prototypes

    Returns:
        dists: (N, K) distances
    """
    return 1.0 - torch.mm(h, protos.T)


def calibrate_tau(h_val, y_val, protos, fpr_target=0.05):
    """
    Calibrate class-conditional thresholds τ_k.
    τ_k = (1 - fpr_target) quantile of intra-class distances on validation set.

    Args:
        h_val: (N, d) validation embeddings
        y_val: (N,) validation labels
        protos: (K, d) prototypes
        fpr_target: target false positive rate (default 0.05 → 95th percentile)

    Returns:
        taus: List[float] of length K
    """
    dists = cosine_dist(h_val, protos)  # (N, K)
    taus = []
    for k in range(protos.shape[0]):
        mask = y_val == k
        if mask.sum() == 0:
            taus.append(float('inf'))
            continue
        d_k = dists[mask, k]
        tau_k = torch.quantile(d_k, 1.0 - fpr_target).item()
        taus.append(tau_k)
    return taus


def predict_open_set(h_test, protos, taus):
    """
    Open-set prediction with prototype-radius rejection.

    Args:
        h_test: (N, d) test embeddings
        protos: (K, d) prototypes
        taus: List[float] of length K

    Returns:
        pred: (N,) predicted labels (-1 = unknown)
        novelty_score: (N,) novelty scores (min cosine distance)
    """
    dists = cosine_dist(h_test, protos)  # (N, K)
    min_dist, nearest = dists.min(dim=1)
    tau_tensor = torch.tensor(taus, dtype=torch.float32)
    tau_nearest = tau_tensor[nearest]
    is_unknown = min_dist > tau_nearest
    pred = nearest.clone()
    pred[is_unknown] = -1  # unknown
    return pred, min_dist  # min_dist = novelty score


Writing models.py


## Eksekusi Eksperimen (run_experiment.py)

In [9]:
%%writefile run_experiment.py
"""
Main experiment runner for SepSpace.
Supports all datasets, methods, and class-holdout protocol.
"""

import os
import json
import argparse
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from data_loader import load_ustc_tfc2016, load_mcfp, load_iscx, split_data
from models import (
    SepSpaceModel, BaselineModel,
    train_stage1, train_stage2,
    get_embeddings, compute_prototypes, calibrate_tau
)
from evaluate import (
    evaluate_prototype_radius, evaluate_msp, evaluate_energy,
    evaluate_mahalanobis, evaluate_lof, print_results
)
from nnfst import evaluate_nnfst


class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_experiment(args):
    """Run a single class-holdout experiment."""
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Load dataset
    print(f"\n{'='*72}")
    print(f"Dataset: {args.dataset}, Novel class: {args.novel_class}")
    print(f"Method: {args.method}, N_cap: {args.n_cap}")
    print(f"{'='*72}")

    if args.dataset == "ustc":
        X, y, class_names, vocab_size = load_ustc_tfc2016(
            args.data_root, args.n_cap, seed=args.seed
        )
    elif args.dataset == "mcfp":
        X, y, class_names, vocab_size = load_mcfp(
            args.data_root, args.n_cap, seed=args.seed
        )
    elif args.dataset.startswith("iscx"):
        _subset_map = {
            "iscx-vpn-app":     ("VPN",    "app"),
            "iscx-vpn-service": ("VPN",    "service"),
            "iscx-nonvpn-app":     ("nonVPN", "app"),
            "iscx-nonvpn-service": ("nonVPN", "service"),
            # legacy aliases kept for backward compat
            "iscx-vpn":    ("VPN",    "app"),
            "iscx-nonvpn": ("nonVPN", "service"),
        }
        subset, label_scheme = _subset_map[args.dataset]
        X, y, class_names, vocab_size = load_iscx(
            args.data_root, subset, label_scheme, args.n_cap, seed=args.seed
        )
    else:
        raise ValueError(f"Unknown dataset: {args.dataset}")

    print(f"Loaded {len(y)} samples, {len(class_names)} classes")
    print(f"Classes: {class_names}")

    # Find novel class index
    if args.novel_class.isdigit():
        novel_idx = int(args.novel_class)
    else:
        if args.novel_class not in class_names:
            raise ValueError(f"Novel class '{args.novel_class}' not in {class_names}")
        novel_idx = class_names.index(args.novel_class)

    novel_name = class_names[novel_idx]
    print(f"Novel class: {novel_name} (idx={novel_idx})")

    # Split data
    (X_train, y_train, X_val, y_val,
     X_test_k, y_test_k, X_novel, y_novel, n_classes) = split_data(
        X, y, novel_idx, seed=args.seed
    )

    print(f"Split: train={len(y_train)}, val={len(y_val)}, "
          f"test_known={len(y_test_k)}, test_novel={len(y_novel)}")

    # Create dataloaders
    train_ds = TrafficDataset(X_train, y_train)
    val_ds = TrafficDataset(X_val, y_val)
    test_k_ds = TrafficDataset(X_test_k, y_test_k)
    novel_ds = TrafficDataset(X_novel, np.zeros(len(y_novel), dtype=np.int64))

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size)
    test_k_loader = DataLoader(test_k_ds, batch_size=args.batch_size)
    novel_loader = DataLoader(novel_ds, batch_size=args.batch_size)

    # Train and evaluate based on method
    results = {}

    if args.method == "ce_msp":
        print("\n[Training CE+MSP baseline]")
        model = BaselineModel(vocab_size, n_classes, d_model=args.d_model).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)
        results = evaluate_msp(model, test_k_loader, novel_loader, device)
        print_results(results, f"CE+MSP | novel={novel_name}")

    elif args.method == "ce_energy":
        print("\n[Training CE+Energy baseline]")
        model = BaselineModel(vocab_size, n_classes, d_model=args.d_model).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)
        results = evaluate_energy(model, test_k_loader, novel_loader, device)
        print_results(results, f"CE+Energy | novel={novel_name}")

    elif args.method == "ce_mahalanobis":
        print("\n[Training CE+Mahalanobis baseline]")
        model = BaselineModel(vocab_size, n_classes, d_model=args.d_model).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)
        results = evaluate_mahalanobis(
            model, train_loader, val_loader, test_k_loader, novel_loader, device
        )
        print_results(results, f"CE+Mahalanobis | novel={novel_name}")

    elif args.method == "ce_pr":
        print("\n[Training CE+PR (no SupCon)]")
        model = SepSpaceModel(vocab_size, n_classes, d_model=args.d_model, d_proj=args.d_proj).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)

        h_train, y_tr = get_embeddings(model, train_loader, device)
        h_val, y_vl = get_embeddings(model, val_loader, device)
        protos = compute_prototypes(h_train, y_tr, n_classes)
        taus = calibrate_tau(h_val, y_vl, protos, fpr_target=0.05)

        results = evaluate_prototype_radius(
            model, test_k_loader, novel_loader, protos, taus, device
        )
        print_results(results, f"CE+PR | novel={novel_name}")

    elif args.method == "sepspace":
        print("\n[Training SepSpace (CE+SupCon+PR)]")
        model = SepSpaceModel(vocab_size, n_classes, d_model=args.d_model, d_proj=args.d_proj).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)
        train_stage2(model, train_loader, args.epochs_joint, args.lr * 0.3, args.lambda_sup, device)

        h_train, y_tr = get_embeddings(model, train_loader, device)
        h_val, y_vl = get_embeddings(model, val_loader, device)
        protos = compute_prototypes(h_train, y_tr, n_classes)
        taus = calibrate_tau(h_val, y_vl, protos, fpr_target=0.05)

        results = evaluate_prototype_radius(
            model, test_k_loader, novel_loader, protos, taus, device
        )
        print_results(results, f"SepSpace | novel={novel_name}")

    elif args.method == "sepspace_frozen":
        print("\n[Training SepSpace (frozen encoder)]")
        model = SepSpaceModel(vocab_size, n_classes, d_model=args.d_model, d_proj=args.d_proj).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)

        # Stage 2 with frozen encoder
        for p in model.encoder.parameters():
            p.requires_grad = False
        for p in model.classifier.parameters():
            p.requires_grad = False

        opt = torch.optim.Adam(model.proj_head.parameters(), lr=args.lr * 0.3)
        model.train()
        for ep in range(args.epochs_joint):
            total_loss = 0
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                z = model.project(X)
                from models import supcon_loss
                loss = supcon_loss(z, y)
                opt.zero_grad()
                loss.backward()
                opt.step()
                total_loss += loss.item()
            if (ep + 1) % 5 == 0:
                print(f"  Stage2 (frozen) ep{ep+1:02d}: loss={total_loss/len(train_loader):.4f}")

        # Unfreeze
        for p in model.encoder.parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True

        h_train, y_tr = get_embeddings(model, train_loader, device)
        h_val, y_vl = get_embeddings(model, val_loader, device)
        protos = compute_prototypes(h_train, y_tr, n_classes)
        taus = calibrate_tau(h_val, y_vl, protos, fpr_target=0.05)

        results = evaluate_prototype_radius(
            model, test_k_loader, novel_loader, protos, taus, device
        )
        print_results(results, f"SepSpace (frozen) | novel={novel_name}")

    elif args.method == "sepspace_lof":
        print("\n[Training SepSpace+LOF]")
        model = SepSpaceModel(vocab_size, n_classes, d_model=args.d_model, d_proj=args.d_proj).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)
        train_stage2(model, train_loader, args.epochs_joint, args.lr * 0.3, args.lambda_sup, device)

        results = evaluate_lof(model, train_loader, test_k_loader, novel_loader, device)
        print_results(results, f"SepSpace+LOF | novel={novel_name}")

    elif args.method == "nnfst":
        print("\n[Training nNFST (CE Stage1 + subspace OSR)]")
        model = BaselineModel(vocab_size, n_classes, d_model=args.d_model).to(device)
        train_stage1(model, train_loader, args.epochs_ce, args.lr, device)
        results = evaluate_nnfst(
            model, train_loader, val_loader, test_k_loader, novel_loader,
            device, n_components=args.nnfst_components, var_ratio=args.nnfst_var_ratio
        )
        print_results(results, f"nNFST | novel={novel_name}")

    else:
        raise ValueError(f"Unknown method: {args.method}")

    # Save results
    output = {
        "dataset": args.dataset,
        "method": args.method,
        "novel_class": novel_name,
        "novel_idx": novel_idx,
        "n_cap": args.n_cap,
        "seed": args.seed,
        "results": results,
        "config": vars(args)
    }

    os.makedirs(args.output_dir, exist_ok=True)
    output_path = os.path.join(
        args.output_dir,
        f"{args.dataset}_{args.method}_{novel_name}_seed{args.seed}.json"
    )
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nResults saved to: {output_path}")

    return results


def main(args_list=None):
    parser = argparse.ArgumentParser(description="SepSpace Experiment Runner")

    # Dataset
    parser.add_argument("--dataset", type=str, required=True,
                        choices=["ustc", "mcfp",
                                 "iscx-vpn-app", "iscx-vpn-service",
                                 "iscx-nonvpn-app", "iscx-nonvpn-service",
                                 "iscx-vpn", "iscx-nonvpn"],  # legacy aliases
                        help="Dataset name")
    parser.add_argument("--data_root", type=str, required=True,
                        help="Path to dataset root")
    parser.add_argument("--novel_class", type=str, required=True,
                        help="Novel class name or index")
    parser.add_argument("--n_cap", type=int, default=2000,
                        help="Max samples per class")

    # Method
    parser.add_argument("--method", type=str, required=True,
                        choices=["ce_msp", "ce_energy", "ce_mahalanobis",
                                 "ce_pr", "sepspace", "sepspace_frozen", "sepspace_lof",
                                 "nnfst"],
                        help="Method to evaluate")

    # Model
    parser.add_argument("--d_model", type=int, default=64,
                        help="Encoder embedding dimension")
    parser.add_argument("--d_proj", type=int, default=32,
                        help="Projection head dimension")

    # Training
    parser.add_argument("--batch_size", type=int, default=256)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--epochs_ce", type=int, default=20,
                        help="Stage 1 epochs")
    parser.add_argument("--epochs_joint", type=int, default=15,
                        help="Stage 2 epochs")
    parser.add_argument("--lambda_sup", type=float, default=0.5,
                        help="SupCon loss weight")

    # nNFST-specific
    parser.add_argument("--nnfst_components", type=int, default=None,
                        help="Fixed PCA components per class for nNFST (None = auto)")
    parser.add_argument("--nnfst_var_ratio", type=float, default=0.95,
                        help="Cumulative variance ratio for nNFST PCA (default 0.95)")

    # Misc
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output_dir", type=str, default="results",
                        help="Output directory for results")

    args = parser.parse_args(args_list)
    run_experiment(args)


if __name__ == "__main__":
    # If in Jupyter, default args can be specified here or passed in list
    import sys
    if 'ipykernel' in sys.modules:
        print("Running in Jupyter notebook mode. Please pass custom args directly, or run through %run magic.")
    else:
        main()

Writing run_experiment.py


In [10]:
import os
import shutil
from scapy.all import wrpcap, IP, TCP

os.makedirs("/content/data/VPN-PCAPS-01", exist_ok=True)
os.makedirs("/content/data/VPN-PCAPs-02", exist_ok=True)

def create_dummy_pcap(filepath, num_packets=2500):
    packets = [IP(src="10.0.0.1", dst="10.0.0.2")/TCP(sport=80, dport=1234) for _ in range(num_packets)]
    wrpcap(filepath, packets)

classes_to_mock = ["vpn_facebook", "vpn_youtube", "vpn_skype", "vpn_spotify", "vpn_aim"]
for i, cls in enumerate(classes_to_mock):
    target_dir = "/content/data/VPN-PCAPS-01" if i % 2 == 0 else "/content/data/VPN-PCAPs-02"
    pcap_name1 = f"{cls}_0.pcap"
    pcap_name2 = f"{cls}_1.pcap"
    pcap_path1 = os.path.join(target_dir, pcap_name1)
    pcap_path2 = os.path.join(target_dir, pcap_name2)

    # Buat file jika belum ada
    if not os.path.exists(pcap_path1):
        create_dummy_pcap(pcap_path1, num_packets=2500)
    if not os.path.exists(pcap_path2):
        create_dummy_pcap(pcap_path2, num_packets=2500)

print("✓ Dataset simulasi siap di /content/data")

cache_dir = "data/preprocessed/iscx_vpn_app"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

from run_experiment import main

main([
    "--dataset", "iscx-vpn-app",
    "--data_root", "/content/data",
    "--novel_class", "facebook",
    "--method", "sepspace",
    "--epochs_ce", "5",
    "--epochs_joint", "5"
])

✓ Dataset simulasi siap di /content/data
Device: cuda

Dataset: iscx-vpn-app, Novel class: facebook
Method: sepspace, N_cap: 2000
Loading ISCX-VPN (app-level) from /content/data
  aim: 1248 units from 2 pcap(s)
  facebook: 1248 units from 2 pcap(s)
  skype: 1248 units from 2 pcap(s)
  spotify: 1248 units from 2 pcap(s)
  youtube: 1248 units from 2 pcap(s)
  Valid classes (≥500): 5 — ['aim', 'facebook', 'skype', 'spotify', 'youtube']
  Balanced N_cap: 1248
Saved to cache: data/preprocessed/iscx_vpn_app
Loaded 6240 samples, 5 classes
Classes: ['aim', 'facebook', 'skype', 'spotify', 'youtube']
Novel class: facebook (idx=1)
Split: train=2996, val=998, test_known=998, test_novel=1248

[Training SepSpace (CE+SupCon+PR)]
  Stage1 ep05: loss=1.3929 acc=0.2517
  Stage2 ep05: loss=4.1459

[SepSpace | novel=facebook]
  AUROC : 0.5000
  DR    : 0.0000
  Acc   : 0.2645
  N1    : 0.0000
  FPR95 : 1.0000
  AUPR  : 0.5557
  OSCR  : 0.0000

Results saved to: results/iscx-vpn-app_sepspace_facebook_seed4

In [11]:
import os
from scapy.all import wrpcap, IP, TCP

# Membuat direktori data simulasi untuk ISCX-VPN
os.makedirs("/content/data/VPN-PCAPS-01", exist_ok=True)
os.makedirs("/content/data/VPN-PCAPs-02", exist_ok=True)

def create_dummy_pcap(filepath, num_packets=2500):
    # Membuat paket IP/TCP yang cukup besar untuk menghasilkan lebih dari 500 units per kelas
    packets = [IP(src="10.0.0.1", dst="10.0.0.2")/TCP(sport=80, dport=1234) for _ in range(num_packets)]
    wrpcap(filepath, packets)

# Membuat beberapa file pcap tiruan yang lebih besar
classes_to_mock = ["vpn_facebook", "vpn_youtube", "vpn_skype", "vpn_spotify", "vpn_aim"]
for i, cls in enumerate(classes_to_mock):
    target_dir = "/content/data/VPN-PCAPS-01" if i % 2 == 0 else "/content/data/VPN-PCAPs-02"
    for file_idx in range(2):  # Membuat 2 file besar per kelas
        pcap_name = f"{cls}_{file_idx}.pcap"
        create_dummy_pcap(os.path.join(target_dir, pcap_name), num_packets=2500)

print("Dataset tiruan (Dummy PCAP) dengan ukuran besar (>500 unit) berhasil dibuat di /content/data/")

Dataset tiruan (Dummy PCAP) dengan ukuran besar (>500 unit) berhasil dibuat di /content/data/
